# Exp #6 — Guidance-Gap generation on Kaggle

Generates 264 images (24 prompts × 1 seed × [standard + 10 γ₂ variants]) in fp16 on CUDA (~6–9h on P100).

**Notebook settings (right panel):** Accelerator → GPU P100, Internet → ON, Persistence → Files only.

## 1. Clone repo
For a private repo, add a GitHub token as a Kaggle Secret named `GH_TOKEN`.

In [ ]:
import os
import pathlib
import subprocess

REPO_DIR = pathlib.Path('/kaggle/working/gm')
BRANCH = 'radu-exp-6'

token = ''
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GH_TOKEN')
except Exception:
    pass
auth = f'{token}@' if token else ''
url = f'https://{auth}github.com/viftode4/generative-modelling.git'

if (REPO_DIR / '.git').exists():
    print('repo present — pulling latest')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--depth', '1', url, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-1'], check=False)

## 2. Install deps
Scoring deps not included — scoring runs locally.

In [ ]:
!pip install -q \
  diffusers==0.37.1 transformers==5.8.1 accelerate safetensors einops torchsde \
  'huggingface_hub>=0.20' 'numpy>=2,<3' tqdm Pillow

In [ ]:
import torch

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Enable GPU accelerator in notebook settings!'

## 3. Run generation (resumable)
Resumable — restart the notebook and already-generated images are skipped.

In [ ]:
OUT = 'outputs/radu_guidance_gap/kaggle_24x1'

!python scripts/run_radu_guidance_gap.py --out {OUT} --device cuda --dry-run | tail -5

In [ ]:
!python scripts/run_radu_guidance_gap.py \
  --out outputs/radu_guidance_gap/kaggle_24x1 \
  --device cuda --size 1024 --steps 50 --lambda-step 49 --gamma1 5.5

## 4. Package for download
Download the tarball from the Output panel, untar into the repo's `outputs/`, and run local scoring scripts.

In [ ]:
import glob
import subprocess

n = len(glob.glob('outputs/radu_guidance_gap/kaggle_24x1/images/*.png'))
print(f'{n} images generated (expect 264 when complete)')
subprocess.run(
    ['tar', 'czf', '/kaggle/working/guidance_gap_kaggle_24x1.tgz',
     '-C', 'outputs/radu_guidance_gap', 'kaggle_24x1'], check=True)
print('wrote /kaggle/working/guidance_gap_kaggle_24x1.tgz')
subprocess.run(['ls', '-lh', '/kaggle/working/guidance_gap_kaggle_24x1.tgz'])

## 5. Resume after timeout
With Persistence on, `/kaggle/working` survives. Run All again — cell 3 skips done images.

```bash
tar xzf ~/Downloads/guidance_gap_kaggle_24x1.tgz -C generative-modelling/outputs/radu_guidance_gap/
```